In [ ]:
# Install required packages
%pip install gnnad tsaug -q


In [ ]:
import warnings
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from gnnad.graphanomaly import GNNAD
from tsaug import TimeWarp, AddNoise, Dropout
import seaborn as sns

warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)

# Constants
context_length = 300
forecast_length = 50
target_columns = [
    'COOLANT_TEMPERATURE ()',
]
time_col = 'ENGINE_RUN_TINE ()'
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata"

In [ ]:
# ============================================================================
# PREPROCESSING (KEEP EXACTLY AS IS)
# ============================================================================

# Load all CSV files
df_list = []
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(f'{path}/{file}', index_col=False)
        df['drive_id'] = file
        df_list.append(df)

print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')


In [ ]:
def remove_zero_variance_columns(df: pd.DataFrame, exclude_cols: list[str] = None) -> pd.DataFrame:
    """
    Compute std of each std-computable column (numeric only)
    """
    if exclude_cols is None:
        exclude_cols = []
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_to_check = [col for col in numeric_cols if col not in exclude_cols]
  
    std_df = df[cols_to_check].std()
    zero_variance_cols = std_df[std_df == 0].index.tolist()
  
    print(f'{len(zero_variance_cols)} columns with zero variance: {zero_variance_cols}')
  
    if len(zero_variance_cols) > 0:
        df = df.drop(columns=zero_variance_cols)
  
    return df


In [ ]:
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame, time_col: str, id_cols: list[str] = None) -> pd.DataFrame:
    """
    Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
    This preserves the overall statistics while removing duplicate entries.
    """
    if id_cols is None:
        id_cols = []
    
    existing_id_cols = [col for col in id_cols if col in df.columns]
    
    group_cols = [time_col] + existing_id_cols
    
    agg_dict = {}
    for col in df.columns:
        if col not in group_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'mean'
            else:
                agg_dict[col] = 'first'
  
    df_clean = df.groupby(group_cols, as_index=False).agg(agg_dict)
  
    return df_clean


In [ ]:
def downsample(df, time_col, source_file_col, downsample_factor=2):
    result_dfs = []
    
    for source_file in df[source_file_col].unique():
        file_df = df[df[source_file_col] == source_file].copy()
        
        if len(file_df) < downsample_factor * 2:
            continue
        
        file_df = file_df.sort_values(time_col).reset_index(drop=True)
        
        # Simple decimation without pre-smoothing
        downsampled = file_df.iloc[::downsample_factor].copy()
        downsampled[time_col] = np.arange(len(downsampled)) * downsample_factor
        
        result_dfs.append(downsampled.reset_index(drop=True))
    
    return pd.concat(result_dfs, ignore_index=True)


In [ ]:
def filter_long_drives(df, id_col='drive_id', min_length=608):
    """Keep only drives long enough for your context window"""
    drive_lengths = df.groupby(id_col).size()
    valid_drives = drive_lengths[drive_lengths >= min_length].index
    
    print(f"Keeping {len(valid_drives)}/{df[id_col].nunique()} drives")
    print(f"Dropped {len(df) - df[df[id_col].isin(valid_drives)].shape[0]} timesteps")
    
    return df[df[id_col].isin(valid_drives)].reset_index(drop=True)


In [ ]:
# Combine all dataframes
data = pd.concat(df_list, ignore_index=True)

# Clean up
print(f"Total samples: {len(data):,}")
print(f"Unique drives: {data['drive_id'].nunique()}")

# remove some useless columns
data = data.drop(columns=['WARM_UPS_SINCE_CODES_CLEARED ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()'])

data = mean_fill_missing_timestamps_and_remove_duplicates(data, time_col=time_col, id_cols=["drive_id"])
data = remove_zero_variance_columns(data, exclude_cols=["drive_id"])
data = downsample(
    data,
    time_col=time_col,
    source_file_col='drive_id',
    downsample_factor=1
)

data = filter_long_drives(data, min_length=context_length + forecast_length)


In [ ]:
def add_cross_channel_features(data, target_columns):
    """
    Engineer features that capture cross-channel relationships.
    Add these as conditional columns.
    """
    # RPM-to-Speed ratio (gear indicator)
    if 'ENGINE_RPM ()' in data.columns and 'VEHICLE_SPEED ()' in data.columns:
        data['RPM_SPEED_RATIO'] = data['ENGINE_RPM ()'] / (data['VEHICLE_SPEED ()'] + 1)
    
    # Throttle-to-Load ratio (efficiency indicator)
    if 'THROTTLE ()' in data.columns and 'ENGINE_LOAD ()' in data.columns:
        data['THROTTLE_LOAD_RATIO'] = data['THROTTLE ()'] / (data['ENGINE_LOAD ()'] + 1)
    
    # Speed-based categories
    if 'VEHICLE_SPEED ()' in data.columns:
        data['IS_IDLE'] = (data['VEHICLE_SPEED ()'] < 5).astype(float)
        data['IS_HIGHWAY'] = (data['VEHICLE_SPEED ()'] > 60).astype(float)
    
    # RPM acceleration
    if 'ENGINE_RPM ()' in data.columns:
        data['RPM_ACCEL'] = data.groupby('drive_id')['ENGINE_RPM ()'].diff().fillna(0)
    
    return data

# Apply cross-channel features before preprocessing
data = add_cross_channel_features(data, target_columns)
print("Added cross-channel features")


In [ ]:
# Ensure data is sorted
data = data.sort_values(["drive_id", time_col]).reset_index(drop=True)

# Get unique drive IDs
unique_drives = data['drive_id'].unique()
n_drives = len(unique_drives)

# Split drives into train/val/test groups (70/15/15)
train_drives = unique_drives[:int(0.70 * n_drives)]
val_drives   = unique_drives[int(0.70 * n_drives):int(0.85 * n_drives)]
test_drives  = unique_drives[int(0.85 * n_drives):]

print(f"Train drives: {len(train_drives)}, Val drives: {len(val_drives)}, Test drives: {len(test_drives)}")

# Create split dataframes
train_data = data[data['drive_id'].isin(train_drives)].copy()
val_data   = data[data['drive_id'].isin(val_drives)].copy()
test_data  = data[data['drive_id'].isin(test_drives)].copy()

print(f"Train shape: {train_data.shape}, Val shape: {val_data.shape}, Test shape: {test_data.shape}")


In [ ]:
# ============================================================================
# CONVERT TO GNNAD FORMAT (MTS: Multivariate Time Series)
# ============================================================================

def drive_to_mts(df_group, targets):
    """Convert drive group to multivariate time series array"""
    return df_group[targets].values.astype(np.float32)

# Convert each drive to MTS format
targets = ['COOLANT_TEMPERATURE ()']

# Concatenate all drives into single MTS arrays
X_train_mts = np.concatenate([drive_to_mts(group, targets) 
                              for _, group in train_data.groupby('drive_id')], axis=0)
X_val_mts   = np.concatenate([drive_to_mts(group, targets) 
                              for _, group in val_data.groupby('drive_id')], axis=0)
X_test_mts  = np.concatenate([drive_to_mts(group, targets) 
                              for _, group in test_data.groupby('drive_id')], axis=0)

print(f"Train MTS shape: {X_train_mts.shape}")
print(f"Val MTS shape: {X_val_mts.shape}")
print(f"Test MTS shape: {X_test_mts.shape}")

# Convert to DataFrames (GNNAD expects DataFrames with column names)
X_train_df = pd.DataFrame(X_train_mts, columns=targets)
X_val_df = pd.DataFrame(X_val_mts, columns=targets)
X_test_df = pd.DataFrame(X_test_mts, columns=targets)

# For GNNAD, we need labels (all zeros for normal data in unsupervised setting)
y_test = pd.Series(np.zeros(len(X_test_df), dtype=int))


In [ ]:
# ============================================================================
# TRAIN GNNAD (Graph Neural Network Anomaly Detection)
# ============================================================================

# Determine device
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")

# Initialize GNNAD model
# slide_win should match context_length, topk controls graph sparsity
model = GNNAD(
    slide_win=context_length,  # Use context_length as sliding window (300)
    topk=10,                   # Number of neighbors in learned graph
    device=device,
    threshold_type="max_validation",  # Use max validation score as threshold
    epoch=50,                   # Number of training epochs
    batch=64,                   # Batch size
    lr=0.001,                   # Learning rate
    embed_dim=64,               # Embedding dimension
    slide_stride=5,             # Stride for sliding window
    validate_ratio=0.1          # Validation split ratio
)

# Combine train and val for training (GNNAD will split internally)
# For unsupervised learning, we train on normal data
X_train_combined = pd.concat([X_train_df, X_val_df], ignore_index=True)

# Fit model (unsupervised on normal drives)
# Note: GNNAD.fit() expects (X_train, X_test, y_test)
# We use X_train_combined for training, X_test_df for testing, y_test for labels
print("Training GNNAD model...")
print(f"Training on {len(X_train_combined)} samples, testing on {len(X_test_df)} samples")
fitted_model = model.fit(X_train_combined, X_test_df, y_test)
print("Training complete!")


In [ ]:
# ============================================================================
# TSAUG AUGMENTATION (Test-time augmentation for robustness)
# ============================================================================

# Create tsaug pipeline
tsaug_pipeline = (
    TimeWarp(n_speed_change=3) * 2 +
    AddNoise(scale=0.02) +
    Dropout(p_array=[0.01, 0.05])
)

# Apply augmentation to test data
print("Applying tsaug augmentation to test data...")
X_test_aug_list = []
for seq in X_test_mts:
    # tsaug expects shape (n_samples, n_features)
    seq_2d = np.expand_dims(seq, 0)  # Shape: (1, n_features)
    aug_seq = tsaug_pipeline.augment(seq_2d)
    X_test_aug_list.append(aug_seq[0])  # Extract augmented sequence

X_test_aug = np.array(X_test_aug_list)
X_test_aug_df = pd.DataFrame(X_test_aug, columns=targets)

print(f"Augmented test shape: {X_test_aug.shape}")


In [ ]:
# ============================================================================
# EVALUATE ANOMALY SCORES
# ============================================================================

# Get anomaly scores from validation set (for threshold)
# We need to evaluate on validation data to get threshold
from gnnad.graphanomaly import parse_data, get_full_err_scores, aggregate_error_scores

# Get validation scores
val_input = parse_data(X_val_df, targets)
val_result = fitted_model._test(fitted_model.model, fitted_model.validate_dataloader)[1]
val_err_scores = get_full_err_scores(val_result, fitted_model.smoothen_error)
_, val_agg_scores = aggregate_error_scores(val_err_scores, topk=fitted_model.topk)

# Get threshold from validation set
thresh = np.max(val_agg_scores)
print(f"Anomaly threshold (max validation): {thresh:.4f}")

# Evaluate on test set (clean)
test_result = fitted_model._test(fitted_model.model, fitted_model.test_dataloader)[1]
test_err_scores = get_full_err_scores(test_result, fitted_model.smoothen_error)
_, test_agg_scores = aggregate_error_scores(test_err_scores, topk=fitted_model.topk)

# Evaluate on augmented test set
# Create a temporary dataloader for augmented data
from gnnad.graphanomaly import TimeDataset
aug_input = parse_data(X_test_aug_df, targets, labels=y_test)
aug_dataset = TimeDataset(aug_input, fitted_model.edge_index, mode="test", config=fitted_model.config)
aug_dataloader = torch.utils.data.DataLoader(
    aug_dataset, batch_size=fitted_model.batch, shuffle=False
)
aug_result = fitted_model._test(fitted_model.model, aug_dataloader)[1]
aug_err_scores = get_full_err_scores(aug_result, fitted_model.smoothen_error)
_, aug_agg_scores = aggregate_error_scores(aug_err_scores, topk=fitted_model.topk)

print(f"Test scores shape: {test_agg_scores.shape}")
print(f"Augmented test scores shape: {aug_agg_scores.shape}")


In [ ]:
# ============================================================================
# VISUALIZE ANOMALY SCORES
# ============================================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot clean test scores
axes[0].plot(test_agg_scores, linewidth=1.5, alpha=0.7, label='Anomaly Score')
axes[0].axhline(thresh, color='red', linestyle='--', linewidth=2, label=f'Threshold ({thresh:.4f})')
axes[0].fill_between(range(len(test_agg_scores)), test_agg_scores, thresh, 
                     where=(test_agg_scores > thresh), alpha=0.3, color='red', label='Anomaly Region')
axes[0].set_title('GDN Anomaly Scores: CAR-OBD Clean Test Data', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time Step', fontsize=12)
axes[0].set_ylabel('Anomaly Score', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot augmented test scores
axes[1].plot(aug_agg_scores, linewidth=1.5, alpha=0.7, label='Anomaly Score (Augmented)')
axes[1].axhline(thresh, color='red', linestyle='--', linewidth=2, label=f'Threshold ({thresh:.4f})')
axes[1].fill_between(range(len(aug_agg_scores)), aug_agg_scores, thresh, 
                     where=(aug_agg_scores > thresh), alpha=0.3, color='red', label='Anomaly Region')
axes[1].set_title('GDN Anomaly Scores: CAR-OBD Augmented Test Data (tsaug)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time Step', fontsize=12)
axes[1].set_ylabel('Anomaly Score', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('gnnad_anomaly_scores.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# COMPUTE METRICS
# ============================================================================

# False Positive Rate (FPR) - percentage of points flagged as anomalies
fpr_clean = (test_agg_scores > thresh).mean()
fpr_aug = (aug_agg_scores > thresh).mean()

print("="*70)
print("ANOMALY DETECTION RESULTS")
print("="*70)
print(f"Threshold (max validation): {thresh:.4f}")
print(f"\nClean Test Data:")
print(f"  - FPR (False Positive Rate): {fpr_clean:.4f} ({fpr_clean*100:.2f}%)")
print(f"  - Max Score: {np.max(test_agg_scores):.4f}")
print(f"  - Mean Score: {np.mean(test_agg_scores):.4f}")
print(f"\nAugmented Test Data (tsaug):")
print(f"  - Detection Rate: {fpr_aug:.4f} ({fpr_aug*100:.2f}%)")
print(f"  - Max Score: {np.max(aug_agg_scores):.4f}")
print(f"  - Mean Score: {np.mean(aug_agg_scores):.4f}")
print("="*70)


In [ ]:
# ============================================================================
# PUBLICATION-READY RESULTS TABLE
# ============================================================================

# Create results DataFrame
# Note: TFT baseline values should be filled from previous runs
results = pd.DataFrame({
    'Method': ['TFT (baseline)', 'GDN+GNNAD (ours)'],
    'CAR-OBD FPR': [0.00, fpr_clean],  # TODO: Fill TFT FPR from previous runs
    'CAR-OBD+tsaug Detection': [0.00, fpr_aug]  # TODO: Fill TFT detection rate
})

# Format for display
results_display = results.copy()
results_display['CAR-OBD FPR'] = results_display['CAR-OBD FPR'].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else str(x))
results_display['CAR-OBD+tsaug Detection'] = results_display['CAR-OBD+tsaug Detection'].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else str(x))

print("\n" + "="*70)
print("PUBLICATION-READY RESULTS TABLE")
print("="*70)
print(results_display.to_string(index=False))
print("="*70)

# Save results
results.to_csv('gnnad_results.csv', index=False)
print("\nResults saved to 'gnnad_results.csv'")


## Citations

**GDN (Graph Deviation Network):**
- Deng, A., & Hooi, B. (2021). Graph neural network-based anomaly detection in multivariate time series. *Proceedings of the AAAI Conference on Artificial Intelligence*, 35(5), 4027-4035.

**GNNAD (Graph Neural Network Anomaly Detection):**
- Buchhorn, K., et al. (2024). GNNAD: Graph Neural Network Anomaly Detection. *F1000Research*.

**tsaug:**
- Time Series Augmentation library for robustness testing.
